# Day 5 — QLoRA Fine-Tuning on a Mini Q&A Dataset

This notebook fine-tunes **`meta-llama/Llama-3.2-1B`** with 4-bit quantization and LoRA adapters. It is designed for Google Colab with a GPU runtime.

**Pipeline:**

```text
Mini Q&A data → prompt formatting → 4-bit base model → LoRA adapters → Trainer
                                                     ↓
                                          loss plot + inference test
                                                     ↓
                                      adapter weights → Hugging Face Hub
```

> QLoRA trains small adapter weights while the quantized base model stays frozen. The output repository contains adapters, not a full copy of the base model.


## Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. Create a Hugging Face access token with write permission if you want to upload adapters.
3. Open the gated model page [`meta-llama/Llama-3.2-1B`](https://huggingface.co/meta-llama/Llama-3.2-1B), request/accept access, and agree to Meta's license.
4. Do not paste tokens into notebook code. Use Colab Secrets named `HF_TOKEN`, or the login widget in the authentication cell.

The model is gated. A token alone is not enough if your Hugging Face account has not been granted access.


In [ ]:
# Install the notebook dependencies. Restart the runtime only if Colab asks you to.
!pip -q install -U transformers datasets peft bitsandbytes accelerate huggingface_hub matplotlib


In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU: Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from huggingface_hub import login

# Recommended: add HF_TOKEN in Colab's Secrets panel, then run this cell.
# If the secret is unavailable, the login widget will ask for it without printing it.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication completed.")
else:
    login()


In [ ]:
from datasets import Dataset

# Small, transparent domain-specific dataset for learning. Replace these examples
# with your own authorized Q&A data for a real experiment.
qa_rows = [
    {"question": "What does an API gateway do?", "answer": "An API gateway is a controlled entry point that routes requests to backend services and can apply authentication, rate limits, validation, and observability."},
    {"question": "Why should secrets stay out of source code?", "answer": "Secrets should stay out of source code because repositories and logs can be copied or exposed. Use environment variables or a managed secret store and rotate leaked credentials."},
    {"question": "What is a database index?", "answer": "A database index is an auxiliary data structure that helps the database find rows faster, usually at the cost of additional storage and write overhead."},
    {"question": "What is retrieval-augmented generation?", "answer": "Retrieval-augmented generation retrieves relevant external context and gives it to a language model so the answer can be grounded in that context."},
    {"question": "What is a health check endpoint?", "answer": "A health check endpoint reports whether a service is running and, when appropriate, whether its critical dependencies are available."},
    {"question": "Why use retries with exponential backoff?", "answer": "Retries with exponential backoff reduce pressure on a temporarily failing dependency by waiting longer between attempts and avoiding an immediate request storm."},
    {"question": "What is least privilege?", "answer": "Least privilege means giving a user, service, or workload only the permissions required for its task and no broader access."},
    {"question": "What does a unit test verify?", "answer": "A unit test verifies a small piece of behavior in isolation so regressions can be detected quickly and failures are easier to diagnose."},
]

dataset = Dataset.from_list(qa_rows).train_test_split(test_size=0.25, seed=SEED)
dataset


In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B"
OUTPUT_DIR = "./day5-qlora-output"
HUB_REPO_ID = "YOUR_USERNAME/day5-llama-3.2-1b-qlora"  # Change this before uploading
MAX_LENGTH = 256

def format_example(row):
    text = (
        "### Instruction\n"
        "Answer the technical question accurately and concisely.\n\n"
        f"### Question\n{row['question']}\n\n"
        f"### Answer\n{row['answer']}"
    )
    return {"text": text}

formatted = dataset.map(format_example)
print(formatted["train"][0]["text"])


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

tokenized = formatted.map(tokenize, batched=True, remove_columns=formatted["train"].column_names)
print(tokenized)


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1
print("Loaded 4-bit base model.")


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    evaluation_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    optim="paged_adamw_8bit",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)


In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete.")
print(train_result.metrics)


In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_points = [(x["step"], x["loss"]) for x in logs if "loss" in x]
eval_points = [(x["step"], x["eval_loss"]) for x in logs if "eval_loss" in x]

plt.figure(figsize=(8, 4))
if train_points:
    x, y = zip(*train_points)
    plt.plot(x, y, marker="o", label="training loss")
if eval_points:
    x, y = zip(*eval_points)
    plt.plot(x, y, marker="s", label="evaluation loss")
plt.title("Day 5 QLoRA training history")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
from transformers import pipeline

# Reload the adapter-backed model through the Trainer object for a quick test.
model.eval()
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

test_question = "What is least privilege?"
prompt = "### Instruction\nAnswer the technical question accurately and concisely.\n\n### Question\n" + test_question + "\n\n### Answer\n"
result = generator(prompt)[0]["generated_text"]
print(result)


In [ ]:
# Change HUB_REPO_ID before running this cell.
if HUB_REPO_ID.startswith("YOUR_USERNAME/"):
    raise ValueError("Update HUB_REPO_ID to your Hugging Face username and repository name first.")

model.push_to_hub(HUB_REPO_ID, private=False)
tokenizer.push_to_hub(HUB_REPO_ID)
print(f"Uploaded adapter and tokenizer to https://huggingface.co/{HUB_REPO_ID}")


## What is uploaded?

QLoRA saves the small LoRA adapter weights and tokenizer. The base model remains referenced by the adapter configuration, so users need permission to download `meta-llama/Llama-3.2-1B`.

## Learning notes

- This tiny dataset demonstrates the mechanics; it is not enough to create a production-quality assistant.
- The model can memorize examples because the dataset is intentionally small.
- Keep a held-out validation set and compare the fine-tuned model against the base model.
- Review generated answers for factual accuracy and safety.
- The notebook uses `evaluation_strategy="epoch"`; if a future Transformers version renames this argument, replace it with the version's current equivalent.
